In [6]:
# 1. Verify the T4 GPU allocation
!nvidia-smi

# 2. Install the necessary PEFT and optimization libraries
!pip install -q transformers datasets peft bitsandbytes accelerate

Fri Jun 12 17:59:09 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   55C    P0             28W /   70W |     809MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [7]:
import json
from datasets import Dataset
from transformers import AutoTokenizer

# 1. Initialize Tokenizer
MODEL_ID = "Salesforce/codegen-350M-mono"
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token

# 2. Open and Flatten Data Safely
with open("train_spider.json", "r", encoding="utf-8") as f:
    spider_train = json.load(f)

# Loop over the raw data and pull out ONLY the flat text fields we need.
# This strips away the nested SQL dictionaries that confuse PyArrow.
flattened_data = []
for entry in spider_train:
    flattened_data.append({
        "question": entry.get("question", ""),
        "db_id": entry.get("db_id", ""),
        "query": entry.get("query", "")  # Holds the golden target SQL string
    })

def formatting_prompts_func(examples):
    texts = []
    for q, db, target in zip(examples['question'], examples['db_id'], examples['query']):
        text = f"### Instruction:\nConvert this question to SQL for database: {db}\n### Question:\n{q}\n### Response:\n{target}<|endoftext|>"
        texts.append(text)
    return {"text": texts}

def tokenize_function(examples):
    model_inputs = tokenizer(examples["text"], truncation=True, max_length=512, padding=False)
    model_inputs["labels"] = model_inputs["input_ids"].copy()
    return model_inputs

# 3. Create Dataset from the clean, flattened list of pure strings
raw_dataset = Dataset.from_list(flattened_data)
text_dataset = raw_dataset.map(formatting_prompts_func, batched=True)
tokenized_dataset = text_dataset.map(tokenize_function, batched=True, remove_columns=text_dataset.column_names)

print("\n" + "="*50)
print(f"📊 Total records compiled for fine-tuning matrices: {len(tokenized_dataset)}")
print("="*50)

tokenizer_config.json:   0%|          | 0.00/240 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/798k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/1.00k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/90.0 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.11M [00:00<?, ?B/s]

Map:   0%|          | 0/7000 [00:00<?, ? examples/s]

Map:   0%|          | 0/7000 [00:00<?, ? examples/s]


📊 Total records compiled for fine-tuning matrices: 7000


In [8]:
!pip install --upgrade torchao

Let's execute Step 4: Initializing the Model and Injecting LoRA Weights.

In this step, we will load the base CodeGen-350M weights in 16-bit precision to save VRAM, and then inject Low-Rank Adaptation (LoRA) matrices specifically targeting the Multi-Head Attention layer query and value projection gates (q_proj and v_proj).

Create a new code cell right below your data compilation cell, paste this block, and run it:

In [9]:
import torch
from transformers import AutoModelForCausalLM
from peft import LoraConfig, get_peft_model, TaskType

print("⚙️ Loading foundational CodeGen weights...")

# 1. Load the base model in FP16
model = AutoModelForCausalLM.from_pretrained(
    "Salesforce/codegen-350M-mono",
    torch_dtype=torch.float16,
    device_map="auto"
)

# 2. Configure the LoRA Bottleneck targeting CodeGen's specific QKV projection
peft_config = LoraConfig(
    r=8,
    lora_alpha=32,
    target_modules=["qkv_proj"], # Updated for CodeGen architecture
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM
)

# 3. Wrap the base model with the LoRA layers
model = get_peft_model(model, peft_config)

print("\n" + "="*50)
print("✅ LoRA Matrix Layer Injection Complete!")
model.print_trainable_parameters()
print("="*50)

⚙️ Loading foundational CodeGen weights...


Loading weights:   0%|          | 0/165 [00:00<?, ?it/s]

[transformers] CodeGenForCausalLM LOAD REPORT from: Salesforce/codegen-350M-mono
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
transformer.h.{0...19}.attn.causal_mask | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



✅ LoRA Matrix Layer Injection Complete!
trainable params: 655,360 || all params: 357,367,808 || trainable%: 0.1834


In [10]:
from transformers import TrainingArguments, Trainer, DataCollatorForSeq2Seq

print("🏋️‍♂️ Setting up training configurations...")

# 1. Define Training Arguments
training_args = TrainingArguments(
    output_dir="./results/codegen_lora_sql",
    per_device_train_batch_size=4,   # Actual memory batch slice size
    gradient_accumulation_steps=4,   # Steps before updating weights (Effective Batch Size = 16)
    learning_rate=2e-4,              # Standard optimized LoRA learning rate
    logging_steps=10,                # Console feedback frequency
    num_train_epochs=3,              # 3 full iterations through your 7,000 samples
    save_strategy="epoch",           # Save model progress states after every epoch
    eval_strategy="no",              # Fix: removed the old 'evaluation_strategy' keyword completely
    fp16=True,                       # Use mixed-precision acceleration
    warmup_ratio=0.03,               # Slowly scale up the learning rate at start
    weight_decay=0.01,               # Pervasive regularization to prevent overfitting
    report_to="none"                 # Keeps execution independent of WandB or external hubs
)

# 2. Set up the dynamic padding collator
data_collator = DataCollatorForSeq2Seq(
    tokenizer,
    pad_to_multiple_of=8,
    return_tensors="pt",
    padding=True
)

# 3. Instantiate the execution Trainer engine
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=data_collator
)

print("🚀 Starting fine-tuning loop! Tracking loss optimization below...")
trainer.train()

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


🏋️‍♂️ Setting up training configurations...
🚀 Starting fine-tuning loop! Tracking loss optimization below...


Step,Training Loss
10,2.836903
20,2.648517
30,2.184613
40,1.574209
50,1.358893
60,1.213592
70,1.157813
80,1.139282
90,1.154166
100,1.086084


TrainOutput(global_step=1314, training_loss=0.8809532540392476, metrics={'train_runtime': 624.0652, 'train_samples_per_second': 33.65, 'train_steps_per_second': 2.106, 'total_flos': 4475138510684160.0, 'train_loss': 0.8809532540392476, 'epoch': 3.0})

In [11]:
# 1. Compress the entire results folder into a single zip archive
!zip -r codegen_lora_sql_checkpoints.zip ./results/codegen_lora_sql

# 2. Use Colab's native utility to trigger an automatic browser download
from google.colab import files
files.download('codegen_lora_sql_checkpoints.zip')

  adding: results/codegen_lora_sql/ (stored 0%)
  adding: results/codegen_lora_sql/checkpoint-876/ (stored 0%)
  adding: results/codegen_lora_sql/checkpoint-876/tokenizer.json (deflated 82%)
  adding: results/codegen_lora_sql/checkpoint-876/adapter_model.safetensors (deflated 7%)
  adding: results/codegen_lora_sql/checkpoint-876/scaler.pt (deflated 64%)
  adding: results/codegen_lora_sql/checkpoint-876/optimizer.pt (deflated 7%)
  adding: results/codegen_lora_sql/checkpoint-876/tokenizer_config.json (deflated 51%)
  adding: results/codegen_lora_sql/checkpoint-876/training_args.bin (deflated 53%)
  adding: results/codegen_lora_sql/checkpoint-876/adapter_config.json (deflated 58%)
  adding: results/codegen_lora_sql/checkpoint-876/scheduler.pt (deflated 61%)
  adding: results/codegen_lora_sql/checkpoint-876/README.md (deflated 66%)
  adding: results/codegen_lora_sql/checkpoint-876/rng_state.pth (deflated 26%)
  adding: results/codegen_lora_sql/checkpoint-876/trainer_state.json (deflated 7

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [12]:
import torch
import json
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

MODEL_ID = "Salesforce/codegen-350M-mono"
CHECKPOINT_PATH = "./results/codegen_lora_sql/checkpoint-1314"

print("🔮 Initializing 5-sample verification pass...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto"
)
model = PeftModel.from_pretrained(base_model, CHECKPOINT_PATH)
model.eval()

# Load the validation data
try:
    with open("dev.json", "r", encoding="utf-8") as f:
        eval_data = json.load(f)
except FileNotFoundError:
    with open("train_spider.json", "r", encoding="utf-8") as f:
        eval_data = json.load(f)

print(f"✅ Data loaded successfully. Running execution on the first 5 samples:\n")

# Run exactly 5 loops
for i, entry in enumerate(eval_data[:5]):
    db = entry.get("db_id", "")
    question = entry.get("question", "")
    ground_truth = entry.get("query", "")

    prompt = f"### Instruction:\nConvert this question to SQL for database: {db}\n### Question:\n{question}\n### Response:\n"
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=64,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
            num_beams=1,
            temperature=0.0
        )

    # Isolate the newly generated SQL string
    full_output = tokenizer.decode(outputs[0], skip_special_tokens=True)
    generated_sql = full_output.split("### Response:\n")[-1].strip()

    print(f"--- [ Sample {i+1} ] ---")
    print(f"DB Context: {db}")
    print(f"Question:   {question}")
    print(f"Expected:   {ground_truth}")
    print(f"Model SQL:  {generated_sql}\n")

🔮 Initializing 5-sample verification pass...


Loading weights:   0%|          | 0/165 [00:00<?, ?it/s]

[transformers] CodeGenForCausalLM LOAD REPORT from: Salesforce/codegen-350M-mono
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
transformer.h.{0...19}.attn.causal_mask | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
[transformers] The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


✅ Data loaded successfully. Running execution on the first 5 samples:

--- [ Sample 1 ] ---
DB Context: concert_singer
Question:   How many singers do we have?
Expected:   SELECT count(*) FROM singer
Model SQL:  SELECT count(*) FROM singer

--- [ Sample 2 ] ---
DB Context: concert_singer
Question:   What is the total number of singers?
Expected:   SELECT count(*) FROM singer
Model SQL:  SELECT count(*) FROM singer

--- [ Sample 3 ] ---
DB Context: concert_singer
Question:   Show name, country, age for all singers ordered by age from the oldest to the youngest.
Expected:   SELECT name ,  country ,  age FROM singer ORDER BY age DESC
Model SQL:  SELECT T2.name ,  T2.country ,  T2.age FROM singer AS T1 JOIN singer_country AS T2 ON T1.singer_id  =  T2.singer_id ORDER BY T1.age_of_birth DESC LIMIT 1

--- [ Sample 4 ] ---
DB Context: concert_singer
Question:   What are the names, countries, and ages for every singer in descending order of age?
Expected:   SELECT name ,  country ,  age FROM si